In [1]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import numpy as np
import pandas as pd

%matplotlib inline
import matplotlib.pyplot as plt

from sklearn.preprocessing import OneHotEncoder, StandardScaler

from config import config
import utils
import objects.datasets

In [10]:
config

{'general': {'project_name': 'Titanic and Houses', 'data_installed': True}, 'paths': {'titanic_targets': 'objects/titanic/gender_submission.csv', 'titanic_train': 'objects/titanic/train.csv', 'titanic_test': 'objects/titanic/test.csv'}}

## Titanic

### Обзор датасета

In [5]:
df_train = pd.read_csv(config.paths.titanic_train)
df_train.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 118.9 KB


In [12]:
df_test = pd.read_csv(config.paths.titanic_test)
df_test.info()

<class 'pandas.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  418 non-null    int64  
 1   Pclass       418 non-null    int64  
 2   Name         418 non-null    str    
 3   Sex          418 non-null    str    
 4   Age          332 non-null    float64
 5   SibSp        418 non-null    int64  
 6   Parch        418 non-null    int64  
 7   Ticket       418 non-null    str    
 8   Fare         417 non-null    float64
 9   Cabin        91 non-null     str    
 10  Embarked     418 non-null    str    
dtypes: float64(2), int64(4), str(5)
memory usage: 52.8 KB


### Очистка

In [13]:
df_train.isna().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [14]:
df_test.isna().sum()

PassengerId      0
Pclass           0
Name             0
Sex              0
Age             86
SibSp            0
Parch            0
Ticket           0
Fare             1
Cabin          327
Embarked         0
dtype: int64

#### Пустые значения в Age

In [15]:
df_train['Initial'] = df_train['Name'].str.extract('([A-Za-z]+)\.')
pd.crosstab(df_train['Initial'], df_train['Sex']).T

Initial,Capt,Col,Countess,Don,Dr,Jonkheer,Lady,Major,Master,Miss,Mlle,Mme,Mr,Mrs,Ms,Rev,Sir
Sex,,,,,,,,,,,,,,,,,
female,0,0,1,0,1,0,1,0,0,182,2,1,0,125,1,0,0
male,1,2,0,1,6,1,0,2,40,0,0,0,517,0,0,6,1


In [16]:
unique_initials = df_train['Initial'].unique().tolist()
initials_to_replace = []
for unique_initial in unique_initials:
    if unique_initial in ['Mlle', 'Mme', 'Ms', 'Dr', 'Major', 'Lady', 'Capt', 'Sir', 'Don', 'Dona']:
        if unique_initial in ['Mlle', 'Mme', 'Ms']:
            initials_to_replace.append('Miss')
        elif unique_initial in ['Dr', 'Major', 'Capt', 'Sir', 'Don']:   
            initials_to_replace.append('Mr')
        else:
            initials_to_replace.append('Mrs')
    elif unique_initial in ['Mr', 'Mrs', 'Miss', 'Master']:
        initials_to_replace.append(unique_initial)
    else:
        initials_to_replace.append('Other')

df_train['Initial'] = df_train['Initial'].replace(unique_initials, initials_to_replace)
pd.crosstab(df_train['Initial'], df_train['Sex']).T

Initial,Master,Miss,Mr,Mrs,Other
Sex,,,,,
female,0,186,1,126,1
male,40,0,528,0,9


In [17]:
mean_age_by_initial = df_train.groupby('Initial')['Age'].mean()

for initial in mean_age_by_initial.keys():
    df_train.loc[(df_train['Age'].isnull()) & (df_train['Initial'] == initial), 'Age'] = mean_age_by_initial[initial]

In [18]:
df_train['Age'].isnull().sum()

np.int64(0)

**Embarked**

In [19]:
df_train['Embarked'] = df_train['Embarked'].fillna('S')
df_train['Embarked'].isna().sum()

np.int64(0)

**Fare (только для test)**

In [20]:
df_test[df_test['Fare'].isna() == True]

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
152,1044,3,"Storey, Mr. Thomas",male,60.5,0,0,3701,NaN,NaN,S


In [21]:
df_test.loc[(df_test['Fare'].isnull()), 'Fare'] = df_test.groupby('Pclass')['Fare'].mean()[3]

In [22]:
df_test.isna().sum()

PassengerId      0
Pclass           0
Name             0
Sex              0
Age             86
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          327
Embarked         0
dtype: int64

### Feature engineering

*Family size* feature

In [23]:
df_train['Family_size'] = df_train['Parch'] + df_train['SibSp']

In [24]:
df_train = df_train.drop(columns=['Parch', 'SibSp', 'Name', 'Ticket', 'Cabin', 'PassengerId'])
df_train.head()

,Survived,Pclass,Sex,Age,Fare,Embarked,Initial,Family_size
0,0,3,male,22.0,7.2500,S,Mr,1
1,1,1,female,38.0,71.2833,C,Mrs,1
2,1,3,female,26.0,7.9250,S,Miss,0
3,1,1,female,35.0,53.1000,S,Mrs,1
4,0,3,male,35.0,8.0500,S,Mr,0


In [32]:
df_test['Pclass'].name

'Pclass'

In [44]:
pclass_encoder = OneHotEncoder(sparse_output=False)
pclass_encoded = pclass_encoder.fit_transform(df_train['Pclass'].to_numpy().reshape(-1, 1))
df_pclass = pd.DataFrame(pclass_encoded, columns=[f'Pclass_{i}' for i in reversed(df_test['Pclass'].unique())])
df_pclass

,Pclass_1,Pclass_2,Pclass_3
0,0.0,0.0,1.0
1,1.0,0.0,0.0
2,0.0,0.0,1.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
886,0.0,1.0,0.0
887,1.0,0.0,0.0
888,0.0,0.0,1.0
889,1.0,0.0,0.0


In [50]:
df_pclass_encoded = utils.make_one_hot_encoding(df_train['Pclass'], drop_first=True)

In [51]:
df_embarked_encoded = utils.make_one_hot_encoding(df_train['Embarked'], drop_first=True)

In [65]:
df_initial_encoded = utils.make_one_hot_encoding(df_train['Initial'], drop_first=True)
df_initial_encoded

,Initial_Miss,Initial_Mr,Initial_Mrs,Initial_Other
0,0.0,1.0,0.0,0.0
1,0.0,0.0,1.0,0.0
2,1.0,0.0,0.0,0.0
3,0.0,0.0,1.0,0.0
4,0.0,1.0,0.0,0.0
...,...,...,...,...
886,0.0,0.0,0.0,1.0
887,1.0,0.0,0.0,0.0
888,1.0,0.0,0.0,0.0
889,0.0,1.0,0.0,0.0


In [54]:
df_family_size_encoded = utils.make_one_hot_encoding(df_train['Family_size'], drop_first=True)

In [35]:
df_train['Sex'] = df_train['Sex'].replace(['male', 'female'], [0, 1])

In [62]:
df_age_scaled = utils.make_standard_scaling(df_train['Age'])
df_age_scaled

,Age
0,-0.587617
1,0.617832
2,-0.286255
3,0.391810
4,0.391810
...,...
886,-0.210914
887,-0.813639
888,-0.598165
889,-0.286255


In [63]:
df_fare_scaled = utils.make_standard_scaling(df_train['Fare'])

In [66]:
concat_dataframe = pd.concat([df_age_scaled, df_fare_scaled, df_initial_encoded, df_embarked_encoded, df_pclass_encoded, df_family_size_encoded], axis=1)
concat_dataframe.insert(0, 'Sex', df_train['Sex'])
concat_dataframe['Survived'] = df_train['Survived']
concat_dataframe

,Sex,Age,Fare,Initial_Miss,Initial_Mr,Initial_Mrs,Initial_Other,Embarked_Q,Embarked_S,Pclass_2,Pclass_3,Family_size_1,Family_size_2,Family_size_3,Family_size_4,Family_size_5,Family_size_6,Family_size_7,Family_size_10,Survived
0,0,-0.587617,-0.502445,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
1,1,0.617832,0.786845,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
2,1,-0.286255,-0.488854,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
3,1,0.391810,0.420730,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
4,0,0.391810,-0.486337,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,0,-0.210914,-0.386671,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
887,1,-0.813639,-0.044381,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
888,1,-0.598165,-0.176263,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0
889,0,-0.286255,-0.044381,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1


In [6]:
type(df_train['Sex'])

pandas.Series

In [38]:
df_preparer = objects.datasets.TitanicDatasetPrepare(config.paths.titanic_train)
df_check = df_preparer.prepare_dataset()
statistics = df_preparer.statistics
# df_check.head(10)
X, y = df_preparer.to_xy()
y

array([0, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1,
       1, 1, 0, 1, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1,
       1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1,
       1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 0,
       1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1, 0, 0, 0,
       0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0,
       0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0, 0, 1, 0, 1, 1, 1, 1, 0, 0,
       1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 1, 0, 0, 0, 1, 1, 0, 1, 0,
       1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1,
       0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 1, 0, 0,
       0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0,
       1, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1,

In [34]:
df_preparer = objects.datasets.TitanicDatasetPrepare(config.paths.titanic_test)
df_check = df_preparer.prepare_dataset(statistics)
df_check.head(10)

,Sex,Age,Fare,Family_size,Pclass_2,Pclass_3,Embarked_Q,Embarked_S,Initial_Miss,Initial_Mr,Initial_Mrs,Initial_Other
0,0,0.353941,-0.490508,-0.560660,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0
1,1,1.295169,-0.507194,0.059127,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0
2,0,2.424644,-0.453112,-0.560660,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
3,0,-0.210796,-0.473739,-0.560660,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0
4,1,-0.587287,-0.400792,0.678913,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0
5,0,-1.189673,-0.462419,-0.560660,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0
6,1,0.015099,-0.494532,-0.560660,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0
7,0,-0.286094,-0.064480,0.678913,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
8,1,-0.888480,-0.502582,-0.560660,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
9,0,-0.662586,-0.162078,0.678913,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0


In [36]:
X = df_preparer.to_xy()
X[:10]

array([[ 0.        ,  0.35394111, -0.49050767, -0.56065994,  0.        ,
         1.        ,  1.        ,  0.        ,  0.        ,  1.        ,
         0.        ,  0.        ],
       [ 1.        ,  1.29516947, -0.50719398,  0.05912667,  0.        ,
         1.        ,  0.        ,  1.        ,  0.        ,  0.        ,
         1.        ,  0.        ],
       [ 0.        ,  2.42464351, -0.45311239, -0.56065994,  1.        ,
         0.        ,  1.        ,  0.        ,  0.        ,  1.        ,
         0.        ,  0.        ],
       [ 0.        , -0.21079592, -0.47373886, -0.56065994,  0.        ,
         1.        ,  0.        ,  1.        ,  0.        ,  1.        ,
         0.        ,  0.        ],
       [ 1.        , -0.58728726, -0.40079158,  0.67891328,  0.        ,
         1.        ,  0.        ,  1.        ,  0.        ,  0.        ,
         1.        ,  0.        ],
       [ 0.        , -1.18967342, -0.46241945, -0.56065994,  0.        ,
         1.        ,  